In [5]:
import pubchempy as pcp
from pathlib import Path
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, rdFingerprintGenerator
import numpy as np
import torch


In [6]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GRAPH_DIR = PROJECT_ROOT / "graph"
NODE_DIR = PROCESSED_DIR / "nodes"
EDGE_DIR = PROCESSED_DIR / "edges"

## Drug Feature

### Generate SMILES of Drugs

In [3]:
drug_nodes = pd.read_csv(NODE_DIR / "drug_nodes.csv")

##### Generate SMILES for each drug node

In [4]:
import json
import time

SMILES_CACHE_PATH = PROCESSED_DIR / "smiles_cache.json"

def load_smiles_cache() -> dict:
    if SMILES_CACHE_PATH.exists():
        return json.loads(SMILES_CACHE_PATH.read_text())
    return {}

def save_smiles_cache(cache: dict) -> None:
    SMILES_CACHE_PATH.write_text(json.dumps(cache, indent=2))


smiles_cache = load_smiles_cache()
smiles = []

for cid in drug_nodes["drug_id"]:
    if cid in smiles_cache:
        smiles.append(smiles_cache[cid])
        continue

    numeric_cid = int(cid.replace("CID", ""))
    resolved = None

    for attempt in range(3):
        try:
            compound = pcp.Compound.from_cid(numeric_cid)
            resolved = compound.smiles
            break
        except Exception as e:
            print(f"{cid}: attempt {attempt + 1}/3 failed ({e})")
            time.sleep(0.5 * (attempt + 1))

    smiles.append(resolved)
    smiles_cache[cid] = resolved
    save_smiles_cache(smiles_cache)
    time.sleep(0.2)

drug_nodes["smiles"] = smiles


CID4168: attempt 1/3 failed (PubChem HTTP Error 502 Bad Gateway)
CID5090: attempt 1/3 failed (PubChem HTTP Error 502 Bad Gateway)


In [5]:
drug_nodes.to_csv(
    NODE_DIR/"drug_nodes_with_smiles.csv",
    index=False
)

### SMILES to RDKit Molecule

In [6]:
drug_nodes = pd.read_csv(
    NODE_DIR / "drug_nodes_with_smiles.csv"
)

In [7]:
def smiles_to_mol(smiles):
    if pd.isna(smiles):
        return None
    return Chem.MolFromSmiles(str(smiles))

drug_nodes["mol"] = drug_nodes["smiles"].apply(smiles_to_mol)

In [8]:
valid = drug_nodes["mol"].notna().sum()
invalid = drug_nodes["mol"].isna().sum()

print(f"Valid molecules: {valid}")
print(f"Invalid molecules: {invalid}")

Valid molecules: 70
Invalid molecules: 0


### Applying Morgan's fingerprint & Compute all Fingerprints

For the valid drugs (see counts printed above) → compute Morgan fingerprints.
For the invalid drugs → assign an all-zero fingerprint.

In [9]:
from rdkit.Chem import rdFingerprintGenerator

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def mol_to_fp(mol):
    if mol is None:
        return np.zeros(2048, dtype=np.float32)

    fp = morgan_gen.GetFingerprint(mol)

    arr = np.zeros((2048,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

drug_features = np.vstack(
    drug_nodes["mol"].apply(mol_to_fp)
)

### Attach to graph

In [10]:
data = torch.load(
    GRAPH_DIR / "heterodata.pt",
    weights_only=False
)

c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
assert len(drug_nodes) == data["drug"].num_nodes, (
    f"drug_nodes has {len(drug_nodes)} rows but graph has "
    f"{data['drug'].num_nodes} drug nodes -- row order must match "
    f"node_index exactly, or features silently attach to the wrong drugs"
)

data["drug"].x = torch.tensor(
    drug_features,
    dtype=torch.float
)

print(data["drug"].x.shape)

torch.Size([70, 2048])


### Save

In [12]:
np.save(
    GRAPH_DIR/"drug_features.npy",
    drug_features
)

In [13]:
reloaded_drug_features = np.load(GRAPH_DIR/"drug_features.npy")
assert np.array_equal(reloaded_drug_features, drug_features)
print("drug_features.npy round-trip verified, shape:", reloaded_drug_features.shape)

drug_features.npy round-trip verified, shape: (70, 2048)


In [14]:
print(data["drug"].x.shape)

torch.Size([70, 2048])


## Gene and Side Effect Feature

Will be create as an embedding layer while constructing the HGNN model

## Save Graph

In [15]:
torch.save(
    data,
    GRAPH_DIR / "heterodata_with_features.pt"
)

In [2]:
import torch

In [8]:
data = torch.load(GRAPH_DIR / "heterodata_with_features.pt", weights_only=False)
print(data[("drug","polypharmacy","drug")].edge_type.max().item() + 1)

c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1301


In [10]:
import pandas as pd
drug_combo = pd.read_csv(RAW_DIR / "drug-combo.csv")
print(drug_combo.columns.tolist())

effect_categories = pd.read_csv(RAW_DIR / "effectcategories.csv")
print(effect_categories.columns.tolist())
print(effect_categories.head())

['STITCH 1', 'STITCH 2', 'Polypharmacy Side Effect', 'Side Effect Name']
['Side Effect', 'Side Effect Name', 'Disease Class']
  Side Effect          Side Effect Name                    Disease Class
0    C0017152      gastric inflammation  gastrointestinal system disease
1    C0027858                   neuroma                  benign neoplasm
2    C0041466                   Typhoid     bacterial infectious disease
3    C0032807  Post thrombotic syndrome    cardiovascular system disease
4    C0033860                 psoriasis     integumentary system disease
